In [0]:
# Databricks notebook source

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DecimalType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable


In [0]:
# COMMAND ----------

spark = (
    SparkSession.builder
    .appName("CarSales_StarSchema")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "8")  # tune for your cluster size
print("✅ Spark session ready — version:", spark.version)

In [0]:
# COMMAND ----------

raw_schema = StructType([
    StructField("Branch_ID",    StringType(),  True),
    StructField("Dealer_ID",    StringType(),  True),
    StructField("Model_ID",     StringType(),  True),
    StructField("Revenue",      LongType(),    True),
    StructField("Units_Sold",   IntegerType(), True),
    StructField("Date_ID",      StringType(),  True),
    StructField("Day",          IntegerType(), True),
    StructField("Month",        IntegerType(), True),
    StructField("Year",         IntegerType(), True),
    StructField("BranchName",   StringType(),  True),
    StructField("DealerName",   StringType(),  True),
    StructField("Product_Name", StringType(),  True),
])

print("✅ Schema defined — columns:", len(raw_schema.fields))


In [0]:
# COMMAND ----------

# ── Autoloader: Ingest new CSV files into staging Delta table
spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .option("cloudFiles.schemaEvolutionMode", "rescue") \
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/fact_to_dim&fact/source_files/schema/") \
    .schema(raw_schema) \
    .load("/Volumes/workspace/fact_to_dim&fact/source_files/") \
    .writeStream \
    .option("checkpointLocation", "/Volumes/workspace/fact_to_dim&fact/source_files/checkpoint/") \
    .option("mergeSchema", "true") \
    .trigger(once=True) \
    .toTable("workspace.`fact_to_dim&fact`.fact_sales_staging") \
    .awaitTermination()


In [0]:
# COMMAND ----------

# ── Read the staging table as the raw batch for downstream processing
df_raw = spark.table("workspace.`fact_to_dim&fact`.fact_sales_staging")


In [0]:
# COMMAND ----------

# ── Display fact_sales_staging table using SQL
display(spark.sql("SELECT * FROM workspace.`fact_to_dim&fact`.fact_sales_staging"))

In [0]:
# COMMAND ----------

# ── 5a: Null checks on critical columns
print("─── NULL CHECK ───────────────────────────────")
critical_cols = ["Branch_ID", "Dealer_ID", "Model_ID", "Date_ID", "Revenue"]
for col in critical_cols:
    null_count = df_raw.filter(F.col(col).isNull()).count()
    print(f"  {col:<15} → {null_count} nulls")


In [0]:
# COMMAND ----------

# ── 5b: Drop rows where ANY key column is null
df_clean = df_raw.dropna(subset=["Branch_ID", "Dealer_ID", "Model_ID", "Date_ID"])


In [0]:
# COMMAND ----------

# ── 5c: Trim whitespace from string columns
string_cols = ["Branch_ID", "Dealer_ID", "Model_ID", "Date_ID",
               "BranchName", "DealerName", "Product_Name"]
for col in string_cols:
    df_clean = df_clean.withColumn(col, F.trim(F.col(col)))


In [0]:
# COMMAND ----------

# ── 5d: Remove exact duplicate rows
df_clean = df_clean.dropDuplicates()

print(f"\n✅ After cleaning — rows: {df_clean.count():,}")


In [0]:
# COMMAND ----------

## 🏢 Cell 6 — Create DimBranch

# Get current max DimBranch_SK from Delta table (fixes monotonically_increasing_id overlap)
try:
    max_sk = spark.table("workspace.`fact_to_dim&fact`.dim_branch") \
        .agg(F.max("DimBranch_SK")).collect()[0][0]
    max_sk = max_sk if max_sk is not None else 0
except Exception:
    max_sk = 0

df_dim_branch = (
    df_clean
    .select("Branch_ID", "BranchName")
    .dropDuplicates(["Branch_ID"])           # deduplicate on natural key
    .withColumn("DimBranch_SK", F.monotonically_increasing_id() + 1 + max_sk)
    .withColumn("Created_At", F.current_timestamp())
    .select("DimBranch_SK", "Branch_ID", "BranchName", "Created_At")
    .orderBy("Branch_ID")
)

print(f"✅ DimBranch rows: {df_dim_branch.count():,}")
df_dim_branch.show(5, truncate=False)


In [0]:
# COMMAND ----------

# ── Write DimBranch as Delta table (creates if not exists)
(
    df_dim_branch.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.`fact_to_dim&fact`.dim_branch")
)
print("✅ DimBranch saved → workspace.`fact_to_dim&fact`.dim_branch")


In [0]:
# COMMAND ----------

## 🏬 Cell 7 — Create DimDealer

# Get current max DimDealer_SK from Delta table (fixes monotonically_increasing_id overlap)
try:
    max_sk = spark.table("workspace.`fact_to_dim&fact`.dim_dealer") \
        .agg(F.max("DimDealer_SK")).collect()[0][0]
    max_sk = max_sk if max_sk is not None else 0
except Exception:
    max_sk = 0

df_dim_dealer = (
    df_clean
    .select("Dealer_ID", "DealerName")
    .dropDuplicates(["Dealer_ID"])
    .withColumn("DimDealer_SK", F.monotonically_increasing_id() + 1 + max_sk)
    .withColumn("Created_At", F.current_timestamp())
    .select("DimDealer_SK", "Dealer_ID", "DealerName", "Created_At")
    .orderBy("Dealer_ID")
)

print(f"✅ DimDealer rows: {df_dim_dealer.count():,}")
df_dim_dealer.show(5, truncate=False)

(
    df_dim_dealer.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.`fact_to_dim&fact`.dim_dealer")
)
print("✅ DimDealer saved → workspace.`fact_to_dim&fact`.dim_dealer")


In [0]:
# COMMAND ----------

## 🚗 Cell 8 — Create DimProduct

# Get current max DimProduct_SK from Delta table (fixes monotonically_increasing_id overlap)
try:
    max_sk = spark.table("workspace.`fact_to_dim&fact`.dim_product") \
        .agg(F.max("DimProduct_SK")).collect()[0][0]
    max_sk = max_sk if max_sk is not None else 0
except Exception:
    max_sk = 0

df_dim_product = (
    df_clean
    .select("Model_ID", "Product_Name")
    .dropDuplicates(["Model_ID"])
    .withColumn("DimProduct_SK", F.monotonically_increasing_id() + 1 + max_sk)
    .withColumn("Created_At", F.current_timestamp())
    .select("DimProduct_SK", "Model_ID", "Product_Name", "Created_At")
    .orderBy("Model_ID")
)

print(f"✅ DimProduct rows: {df_dim_product.count():,}")
df_dim_product.show(5, truncate=False)

(
    df_dim_product.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.`fact_to_dim&fact`.dim_product")
)
print("✅ DimProduct saved → workspace.`fact_to_dim&fact`.dim_product")


In [0]:
# COMMAND ----------

## 📅 Cell 9 — Create DimDate (with full calendar enrichment)

# Get current max DimDate_SK from Delta table (fixes monotonically_increasing_id overlap)
try:
    max_sk = spark.table("workspace.`fact_to_dim&fact`.dim_date") \
        .agg(F.max("DimDate_SK")).collect()[0][0]
    max_sk = max_sk if max_sk is not None else 0
except Exception:
    max_sk = 0

df_dim_date = (
    df_clean
    .select("Date_ID", "Day", "Month", "Year")
    .dropDuplicates(["Date_ID"])
    # ── Build a proper date column
    .withColumn(
        "Full_Date",
        F.to_date(
            F.concat_ws("-", F.col("Year"), F.col("Month"), F.col("Day")),
            "yyyy-M-d"
        )
    )
    # ── Enrich with derived calendar attributes
    .withColumn("Quarter",       F.quarter(F.col("Full_Date")))
    .withColumn("Day_Of_Week",   F.dayofweek(F.col("Full_Date")))   # 1=Sun … 7=Sat
    .withColumn("Day_Name",      F.date_format(F.col("Full_Date"), "EEEE"))
    .withColumn("Month_Name",    F.date_format(F.col("Full_Date"), "MMMM"))
    .withColumn("Is_Weekend",
                F.when(F.col("Day_Of_Week").isin([1, 7]), True).otherwise(False))
    .withColumn("DimDate_SK",    F.monotonically_increasing_id() + 1 + max_sk)
    .withColumn("Created_At",    F.current_timestamp())
    .select(
        "DimDate_SK", "Date_ID", "Full_Date",
        "Day", "Month", "Year", "Quarter",
        "Day_Of_Week", "Day_Name", "Month_Name", "Is_Weekend",
        "Created_At"
    )
    .orderBy("Full_Date")
)

print(f"✅ DimDate rows: {df_dim_date.count():,}")
df_dim_date.show(5, truncate=False)

(
    df_dim_date.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.`fact_to_dim&fact`.dim_date")
)
print("✅ DimDate saved → workspace.`fact_to_dim&fact`.dim_date")


In [0]:
# COMMAND ----------

# ── Re-read dimension tables from Delta (single source of truth for SKs)
dim_branch_sk  = spark.table("workspace.`fact_to_dim&fact`.dim_branch") \
                      .select("Branch_ID",  "DimBranch_SK")
dim_dealer_sk  = spark.table("workspace.`fact_to_dim&fact`.dim_dealer") \
                      .select("Dealer_ID",  "DimDealer_SK")
dim_product_sk = spark.table("workspace.`fact_to_dim&fact`.dim_product") \
                      .select("Model_ID",   "DimProduct_SK")
dim_date_sk    = spark.table("workspace.`fact_to_dim&fact`.dim_date") \
                      .select("Date_ID",    "DimDate_SK")


In [0]:
# COMMAND ----------

# ── Join all dimension SKs onto the clean source
df_fact_sales = (
    df_clean
    .join(dim_branch_sk,  on="Branch_ID",  how="left")
    .join(dim_dealer_sk,  on="Dealer_ID",  how="left")
    .join(dim_product_sk, on="Model_ID",   how="left")
    .join(dim_date_sk,    on="Date_ID",    how="left")
    # ── Surrogate key for the fact row itself
    .withColumn(
        "Sale_SK",
        F.row_number().over(
            Window.orderBy("Date_ID", "Branch_ID", "Dealer_ID")
        )
    )
    .withColumn("Created_At", F.current_timestamp())
    # ── Keep only the fact columns (SKs as FKs + measures)
    .select(
        "Sale_SK",
        "DimBranch_SK",
        "DimDealer_SK",
        "DimProduct_SK",
        "DimDate_SK",
        # Natural keys kept for traceability (optional — remove in strict DW)
        "Branch_ID", "Dealer_ID", "Model_ID", "Date_ID",
        # ── MEASURES
        "Revenue",
        "Units_Sold",
        "Created_At"
    )
)

print(f"✅ FactSales rows: {df_fact_sales.count():,}")
df_fact_sales.show(5, truncate=False)

(
    df_fact_sales.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.`fact_to_dim&fact`.fact_sales")
)
print("✅ FactSales saved → workspace.`fact_to_dim&fact`.fact_sales")


In [0]:
# COMMAND ----------

## ✅ Cell 11 — Verification: Row Count & Sample Joins

print("═" * 55)
print("  STAR SCHEMA — TABLE SUMMARY")
print("═" * 55)

tables = {
    "DimBranch":  "workspace.`fact_to_dim&fact`.dim_branch",
    "DimDealer":  "workspace.`fact_to_dim&fact`.dim_dealer",
    "DimProduct": "workspace.`fact_to_dim&fact`.dim_product",
    "DimDate":    "workspace.`fact_to_dim&fact`.dim_date",
    "FactSales":  "workspace.`fact_to_dim&fact`.fact_sales",
}
for name, table in tables.items():
    count = spark.table(table).count()
    print(f"  {name:<15} → {count:>6,} rows")

print("═" * 55)

In [0]:
print("═" * 55)

# COMMAND ----------

# Load all tables and display
dim_branch  = spark.table("workspace.`fact_to_dim&fact`.dim_branch")
dim_dealer  = spark.table("workspace.`fact_to_dim&fact`.dim_dealer")
dim_product = spark.table("workspace.`fact_to_dim&fact`.dim_product")
dim_date    = spark.table("workspace.`fact_to_dim&fact`.dim_date")
fact_sales  = spark.table("workspace.`fact_to_dim&fact`.fact_sales")

display(dim_branch)
display(dim_dealer)
display(dim_product)
display(dim_date)
display(fact_sales)